In [ ]:
# Chapter 14: Deep Computer Vision Using Convolutional Neural Networks

## Global Imports

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from functools import partial
import tensorflow_datasets as tfds

## The Architecture of the Visual Cortex
Convolutional Neural Networks (CNNs) were inspired by the study of the visual cortex. David Hubel and Torsten Wiesel discovered that neurons in the visual cortex have a small receptive field, meaning they react only to a limited region of the visual field. These neurons are arranged in layers: lower-level neurons detect simple structures (lines, curves), while higher-level neurons combine these into complex patterns (faces, objects). This hierarchical structure is the foundation of CNNs.

<p align="left"><img src="../fig/figure14.1.png" width="45%"></p>

## Convolutional Layers
The most important building block of a CNN is the Convolutional Layer. Unlike dense layers where every neuron connects to every input, neurons in a convolutional layer connect only to pixels within their receptive field (e.g., a $3 \times 3$ window).

<p align="left"><img src="../fig/figure14.2.png" width="45%"></p>
<p align="left"><img src="../fig/figure14.3.png" width="45%"></p>
<p align="left"><img src="../fig/figure14.4.png" width="45%"></p>

### Filters (Kernels)
A layer learns a set of filters (or kernels). A filter might learn to detect vertical lines, while another detects horizontal lines. The layer slides (convolves) these filters across the input image to produce feature maps, which highlight the areas where the filter's pattern was detected.

<p align="left"><img src="../fig/figure14.5.png" width="45%"></p>

### Stride and Padding
- Stride: The distance between two consecutive receptive fields. A larger stride reduces the output dimensionality.
- Padding: To keep the output size the same as the input (or to handle borders), zeros are added around the inputs. padding="SAME" uses zero padding to maintain size (if stride=1), while padding="VALID" means no padding (output may be smaller).

<p align="left"><img src="../fig/figure14.6.png" width="45%"></p>

Implementation in Keras:

In [2]:
images = tf.random.uniform(shape=[2, 28, 28, 3]) # Batch of 2 images

# 32 filters, 3x3 kernel, stride 1, padding 'same', ReLU activation
conv_layer = keras.layers.Conv2D(filters=32, kernel_size=3, strides=1,
                                 padding="same", activation="relu")
output = conv_layer(images)

## Pooling Layers
Pooling layers are used to subsample (shrink) the input image to reduce the computational load, memory usage, and the number of parameters (reducing overfitting).
- Max Pooling: Outputs the maximum value in each window. It is very effective at preserving the strongest features while discarding irrelevant details.
- Average Pooling: Computes the mean value. Less common than max pooling today but used in specific architectures.
- Global Average Pooling: Computes the mean of the entire feature map, outputting a single value per feature map. Often used as the final layer before classification.

<p align="left"><img src="../fig/figure14.8.png" width="45%"></p>
<p align="left"><img src="../fig/figure14.9.png" width="45%"></p>

In [3]:
# Max pooling with a 2x2 pool size and stride 2
max_pool = keras.layers.MaxPool2D(pool_size=2)

# Global Average Pooling (often used at the end of a CNN)
global_avg_pool = keras.layers.GlobalAvgPool2D()

## CNN Architectures
The chapter reviews the evolution of famous CNN architectures:
- LeNet-5 (1998): Used for MNIST. Introduced the sequence of Convolution $\rightarrow$ Pooling $\rightarrow$ Convolution $\rightarrow$ Pooling $\rightarrow$ Dense.
- AlexNet (2012): Deep and wide CNN that won the ImageNet challenge. It introduced ReLU, Dropout, and Data Augmentation.
- GoogLeNet (2014): Introduced the Inception Module, which applies multiple filter sizes ($1 \times 1, 3 \times 3, 5 \times 5$) simultaneously and concatenates their outputs. It is much deeper than AlexNet but has fewer parameters.
- VGGNet (2014): Known for its simplicity, using only $3 \times 3$ filters and $2 \times 2$ pooling repeated many times.
- ResNet (2015): Solved the vanishing gradient problem in very deep networks (e.g., 152 layers) using Skip Connections (Residual Units). The signal feeds into a layer and is also added directly to the layer's output ($y = f(x) + x$).
- Xception (2016): Replaces Inception modules with Depthwise Separable Convolutions (spatially convolving each channel separately, then using $1 \times 1$ convolution).
- SENet (2017): Adds a "Squeeze-and-Excitation" block to recalibrate the importance of feature maps.

### Implementing a ResNet-34 Using Keras
We can build a Residual Network (ResNet-34) from scratch. We first define a ResidualUnit layer that implements the skip connection.

<p align="left"><img src="../fig/figure14.11.png" width="45%"></p>

In [5]:
import tensorflow as tf
from tensorflow import keras
from functools import partial

# PERBAIKAN 1: Ubah padding="SAME" menjadi padding="same"
DefaultConv2D = partial(keras.layers.Conv2D, kernel_size=3, strides=1,
                        padding="same", use_bias=False)

class ResidualUnit(keras.layers.Layer):
    def __init__(self, filters, strides=1, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.activation = keras.activations.get(activation)
        self.main_layers = [
            DefaultConv2D(filters, strides=strides),
            keras.layers.BatchNormalization(),
            self.activation,
            DefaultConv2D(filters),
            keras.layers.BatchNormalization()]
        self.skip_layers = []
        if strides > 1:
            self.skip_layers = [
                DefaultConv2D(filters, kernel_size=1, strides=strides),
                keras.layers.BatchNormalization()]

    def call(self, inputs):
        Z = inputs
        for layer in self.main_layers:
            Z = layer(Z)
        skip_Z = inputs
        for layer in self.skip_layers:
            skip_Z = layer(skip_Z)
        return self.activation(Z + skip_Z)

# Building ResNet-34
model = keras.models.Sequential()

# PERBAIKAN 2: Gunakan layer Input secara eksplisit (Best Practice di Keras baru)
model.add(keras.layers.Input(shape=[224, 224, 3]))

model.add(DefaultConv2D(64, kernel_size=7, strides=2))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Activation("relu"))

# PERBAIKAN 3: Ubah padding="SAME" menjadi padding="same"
model.add(keras.layers.MaxPool2D(pool_size=3, strides=2, padding="same"))

prev_filters = 64
for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
    strides = 1 if filters == prev_filters else 2
    model.add(ResidualUnit(filters, strides=strides))
    prev_filters = filters

model.add(keras.layers.GlobalAvgPool2D())
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(10, activation="softmax"))

# Cek apakah model berhasil dibangun
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit (ResidualUnit)    │ (None, 56, 56, 64)     │        74,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_1 (ResidualUnit)  │ (None, 56, 56, 64)     │        74,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_2 (ResidualUnit)  │ (None, 56, 56, 64)     │        74,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_3 (ResidualUnit)  │ (None, 28, 28, 128)    │       230,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_4 (ResidualUnit)  │ (None, 28, 28, 128)    │       295,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_5 (ResidualUnit)  │ (None, 28, 28, 128)    │       295,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_6 (ResidualUnit)  │ (None, 28, 28, 128)    │       295,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_7 (ResidualUnit)  │ (None, 14, 14, 256)    │       920,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_8 (ResidualUnit)  │ (None, 14, 14, 256)    │     1,181,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_9 (ResidualUnit)  │ (None, 14, 14, 256)    │     1,181,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_10 (ResidualUnit) │ (None, 14, 14, 256)    │     1,181,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_11 (ResidualUnit) │ (None, 14, 14, 256)    │     1,181,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_12 (ResidualUnit) │ (None, 14, 14, 256)    │     1,181,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_13 (ResidualUnit) │ (None, 7, 7, 512)      │     3,676,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_14 (ResidualUnit) │ (None, 7, 7, 512)      │     4,722,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_unit_15 (ResidualUnit) │ (None, 7, 7, 512)      │     4,722,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,306,826 (81.28 MB)

 Trainable params: 21,289,802 (81.21 MB)

 Non-trainable params: 17,024 (66.50 KB)

## Using Pretrained Models from Keras
Instead of training from scratch, you can load state-of-the-art models pretrained on ImageNet (Transfer Learning). Keras includes models like ResNet50, InceptionV3, Xception, etc.

In [6]:
# Load ResNet50 pretrained on ImageNet
model = keras.applications.resnet50.ResNet50(weights="imagenet")

# Resize images to match model input (224x224)
images_resized = tf.image.resize(images, [224, 224])

# Preprocess inputs (scaling specific to the model)
inputs = keras.applications.resnet50.preprocess_input(images_resized * 255)

# Make predictions
Y_proba = model.predict(inputs)

# Decode predictions to human-readable class names
top_K = keras.applications.resnet50.decode_predictions(Y_proba, top=3)
for image_index in range(len(images)):
    print(f"Image #{image_index}")
    for class_id, name, y_proba in top_K[image_index]:
        print(f"  {class_id} - {name} {y_proba*100:.2f}%")

102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Image #0
  n04476259 - tray 60.14%
  n03871628 - packet 7.66%
  n02999410 - chain 5.03%
Image #1
  n03871628 - packet 34.03%
  n07745940 - strawberry 12.54%
  n03089624 - confectionery 9.56%


## Transfer Learning for Custom Tasks
If you want to classify images into categories not present in ImageNet (e.g., specific flowers), you can use a pretrained model as a feature extractor.
1. Load the base model without the top layers (include_top=False).
2. Add your own layers (Global Average Pooling + Dense).
3. Freeze the base layers initially to train only your new layers.
4. Unfreeze and fine-tune with a low learning rate.

In [7]:
# Load Xception base without top layer
base_model = keras.applications.xception.Xception(weights="imagenet",
                                                  include_top=False)
avg = keras.layers.GlobalAveragePooling2D()(base_model.output)
output = keras.layers.Dense(10, activation="softmax")(avg) # 10 classes example
model = keras.models.Model(inputs=base_model.input, outputs=output)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Compile and train (warm up top layers)
optimizer = keras.optimizers.SGD(learning_rate=0.2, momentum=0.9, decay=0.01)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
# history = model.fit(...) # Fit on your dataset

# Unfreeze for fine-tuning
for layer in base_model.layers:
    layer.trainable = True

# Recompile with a lower learning rate
optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, decay=0.001)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
# history = model.fit(...)

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


## Classification and Localization
To not only classify an object but also draw a bounding box around it, you can add a second output head to the network for regression (predicting coordinates $x, y, w, h$).

In [8]:
base_model = keras.applications.xception.Xception(weights="imagenet",
                                                  include_top=False)
avg = keras.layers.GlobalAveragePooling2D()(base_model.output)

# Classification Output
class_output = keras.layers.Dense(10, activation="softmax")(avg)

# Localization Output (4 coordinates)
loc_output = keras.layers.Dense(4)(avg)

model = keras.models.Model(inputs=base_model.input,
                           outputs=[class_output, loc_output])
model.compile(loss=["sparse_categorical_crossentropy", "mse"],
              loss_weights=[0.8, 0.2], # Weight the losses
              optimizer=optimizer, metrics=["accuracy"])

## Object Detection
For detecting multiple objects, we cannot simply add 4 outputs per object because the number of objects varies. Common approaches include:
- You Only Look Once (YOLO): Splits the image into a grid. Each grid cell predicts bounding boxes and class probabilities. It is extremely fast and suitable for real-time detection.
- IoU (Intersection over Union): Metric to measure how well a predicted box overlaps with the ground truth.
- NMS (Non-Max Suppression): A post-processing step to eliminate duplicate bounding boxes for the same object, keeping only the one with the highest confidence.

## Semantic Segmentation
The goal is to classify every single pixel in the image (e.g., distinguishing road, car, pedestrian, sidewalk pixels).
- Fully Convolutional Networks (FCN): Replaces the dense layers at the top of a CNN with convolutional layers.
- Upsampling: To get a dense pixel map (same size as input) from the small feature maps produced by the CNN, we need to upsample.
- Transposed Convolution: A learnable upsampling layer that can increase the spatial dimensions of the input.

In [9]:
# Converting a Pretrained CNN to FCN with Upsampling
base_model = keras.applications.xception.Xception(weights="imagenet",
                                                  include_top=False)
# Add upsampling layers to restore image size
# (Simplified example logic)
input_ = keras.layers.Input(shape=[224, 224, 3])
# ... pass through base model ...
# ... add Conv2DTranspose layers to upscale ...

The chapter concludes by highlighting that while standard CNNs are powerful, complex tasks like detection and segmentation rely on specialized architectures (like YOLO or U-Net) that build upon these fundamental blocks.